STANDALONE MODEL EVALUATION

This notebook answers a question the combined evaluation in `Final_Model.ipynb` doesn't answer on its own:

- how good is each model by itself, and how much does combining them actually help?

`SIZOKUTHOLA_Final_Dataset.csv` already contains each model's individual prediction for every row (`iso_prediction`, `kmeans_prediction`), the combined result (`final_prediction`),

and the true label (`insider_threat`). That means each model can be scored separately without retraining anything.

In [5]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

df = pd.read_csv(r"C:\Users\Admin\SIZOKUTHOLA_Final_Dataset.csv", low_memory=False)
y_true = df["insider_threat"]

print("Total rows:", len(df))
print("Total known insider threat cases:", y_true.sum())

Total rows: 50000
Total known insider threat cases: 2170


1. A reusable evaluation function

Same metrics used in `Final_Model.ipynb`, applied to any prediction column.

In [7]:
def evaluate(name, y_pred):

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)

    print(f"=== {name} ===")
    print(f"Accuracy : {acc:.5f}")
    print(f"Precision: {prec:.5f}")
    print(f"Recall   : {rec:.5f}")
    print(f"F1 Score : {f1:.5f}")
    print("Confusion matrix:")
    print(cm)
    print()

    return {"name": name, "accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "confusion_matrix": cm}

2. Isolation Forest, evaluated on its own

In [9]:
iso_results = evaluate("Isolation Forest ALONE", df["iso_prediction"])

=== Isolation Forest ALONE ===
Accuracy : 0.97660
Precision: 1.00000
Recall   : 0.46083
F1 Score : 0.63091
Confusion matrix:
[[47830     0]
 [ 1170  1000]]



3. K-Means, evaluated on its own

In [ ]:
kmeans_results = evaluate("K-Means ALONE", df["kmeans_prediction"])

4. The combined model (Isolation Forest OR K-Means)

This is the same `final_prediction` column evaluated in `Final_Model.ipynb`.

In [14]:
combined_results = evaluate("Combined (Isolation Forest OR K-Means)", df["final_prediction"])

=== Combined (Isolation Forest OR K-Means) ===
Accuracy : 0.97662
Precision: 1.00000
Recall   : 0.46129
F1 Score : 0.63135
Confusion matrix:
[[47830     0]
 [ 1169  1001]]



5. Does K-Means actually add anything?

Accuracy/precision/recall on their own don't show *which* threat cases each model catches. This checks the overlap directly: of the threats K-Means catches, how many did Isolation Forest already catch too?

In [17]:
threats = df[df["insider_threat"] == 1]

iso_hit = threats["iso_prediction"] == 1
km_hit = threats["kmeans_prediction"] == 1

both = (iso_hit & km_hit).sum()
iso_only = (iso_hit & ~km_hit).sum()
km_only = (~iso_hit & km_hit).sum()
neither = (~iso_hit & ~km_hit).sum()

print("Total known threat cases         :", len(threats))
print("Caught by BOTH models            :", both)
print("Caught by Isolation Forest ONLY  :", iso_only)
print("Caught by K-Means ONLY           :", km_only)
print("Missed by BOTH                   :", neither)
print()
print(f"K-Means contributes {km_only} unique catch(es) that Isolation Forest alone would have missed.")
print(f"Union total (either model): {both + iso_only + km_only} -- matches combined true positives above.")

Total known threat cases         : 2170
Caught by BOTH models            : 335
Caught by Isolation Forest ONLY  : 665
Caught by K-Means ONLY           : 1
Missed by BOTH                   : 1169

K-Means contributes 1 unique catch(es) that Isolation Forest alone would have missed.
Union total (either model): 1001 -- matches combined true positives above.


6. Summary

| Model | Accuracy | Precision | Recall | F1 | Threats caught |
|---|---|---|---|---|---|
| Isolation Forest alone | 97.66% | 100% | 46.08% | 0.631 | 1,000 / 2,170 |
| K-Means alone | 96.33% | 100% | 15.48% | 0.268 | 336 / 2,170 |
| Combined (OR rule) | 97.66% | 100% | 46.13% | 0.631 | 1,001 / 2,170 |

Conclusion: Isolation Forest does almost all of the detection work on its own. K-Means is a much weaker detector in isolation (15.48% recall vs 46.08%), and of the 336 threats it does catch, 335 overlap with what Isolation Forest already found. Combining the two models adds exactly **one** unique true positive over Isolation Forest alone -- a real but very marginal improvement, not a large one.

This is a useful, honest result: it shows the combination is *not redundant with zero benefit*, but it also shows Isolation Forest is the model actually carrying the system. A fair write-up should say this directly, rather than implying both models contribute equally.